## Ingest Drivers Data (Nested JSON to Delta Lake)

This notebook reads the **drivers.json** file from the landing volume and loads it into a Bronze Delta table.

**What's special about this file?** The drivers JSON has a **nested structure** - the `name` field contains an inner object with `givenName` and `familyName`. This requires a nested schema definition.

**Steps:**
1. Load config
2. Define nested schema
3. Read JSON (Read API)
4. Add metadata columns
5. Write to Delta table (Write API)

#### Loading Configuration
We import shared variables and helper functions from the `00-common` folder so we don't repeat code.

In [0]:
%run ../00-common/01.environment-config 


In [0]:
%run ../00-common/02.bronze_helpers 

#### Setting Variables
We define the file path and table name in variables so they're easy to change later.

In [0]:
source_file = f'{loding_folder_path}/drivers.json'
table_name = f'{catalog_name}.{bronze_schema}.drivers'

#### Schema (Nested)

The JSON has a **nested structure** - one object inside another:
```json
{
  "driverId": "hamilton",
  "name": { "givenName": "Lewis", "familyName": "Hamilton" },
  "dateOfBirth": "1985-01-07",
  "nationality": "British"
}
```

To handle this, we create **two schemas**:
- `name_schema` - defines the inner object (givenName, familyName)
- `driver_schema` - the main schema that includes `name_schema` as a nested field

This is how Spark handles **objects inside objects** - you nest one `StructType` inside another.

In [0]:
from pyspark.sql.types import *
name_schema = StructType([
  StructField('givenName', StringType(), True),
  StructField('familyName', StringType(), True)])
drivers_schema = StructType([
  StructField('driverId', StringType(), True),
  StructField('name', name_schema),
  StructField('dateOfBirth', DateType(), True),
  StructField('nationality', StringType(), True),
  StructField('URL', StringType(), True)])

#### Read API
We use `spark.read` to load the JSON file:
- `.format('json')` - file type is JSON
- `.schema(driver_schema)` - apply our nested schema
- `.option('mode', 'FAILFAST')` - stop immediately if data doesn't match the schema
- `.load(source_file)` - path to the file

After reading, the `name` column becomes a **struct** (nested object) that you can access as `name.givenName` or `name.familyName`.

In [0]:
drivers_df = (
  spark.read
   .format ('json')
   .schema(drivers_schema)  # Verify schema data types or remove to infer automatically
   .option('mode', 'FAILFAST')
    .load(source_file))


             

#### Metadata
We call `add_ingestion_metadata()` to add two tracking columns:
- `ingestion_timestamp` - when the data was loaded
- `source_file` - which file the row came from

In [0]:
drivers_final_df = add_ingestion_metadata(drivers_df)

#### Writing Delta Table
We save the final DataFrame to `formula1.bronze.drivers`:
- `.mode('overwrite')` - replace the table completely each run
- `.format('delta')` - Delta format (versioning, fast queries)
- `.saveAsTable(table_name)` - register in Unity Catalog so it's queryable with SQL

In [0]:
(
drivers_final_df
    .write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable(table_name)
)

In [0]:
display (spark.table(table_name))